# EII — Phase 3: Change Point Detection

Applies change point detection to the annual EII time series of each hexagonal
cell to identify structural breaks in interface connectivity.

**Effective analysis period:** 1986–2023 (38 years).  
1985 and 2024 excluded due to truncated post-classification filters in
MapBiomas Collection 10.1 (ATBD Section 3.4.3.3).

**Two modes run in parallel:**
- Mode A: Binseg with n_bkps=1 — exactly 1 break per cell (primary, for mapping)
- Mode B: Pelt with pen=3.0 — automatic detection (secondary, robustness check)

**Penalty selection for Mode B (empirically validated on 500-cell subsample):**
- pen=1.5 → over-segments: 45% cells with 3+ breaks
- pen=3.0 → balanced: 14% no break, 34% one, 50% two, 2% three+
- pen=6.0 → under-segments: 75% forced to 1 break

**Series analysed:** EII only (Paper 1 scope).

**Outputs per cell:**
- `cp_year_A`: year of dominant structural break (Mode A)
- `eii_before`, `eii_after`, `delta_eii`: mean EII before/after break and difference
- `cp_n_B`: number of breaks detected (Mode B)
- `cp_years_B`: break year(s) under Mode B (semicolon-separated)
- `valid`: True if cell had sufficient valid observations

---
## 1. Configuration — edit only this cell

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Consolidated annual EII dataset from Phase 2
EII_CSV = r"D:\Modelo_LAPIG\phase2_annual\eii_HEX20_annual.csv"

# Primary hexagonal grid shapefile (for spatial outputs)
GRID_SHAPEFILE = r"D:\Modelo_LAPIG\grids\hex_20000ha.shp"

# Output folder
OUTPUT_FOLDER = r"D:\Modelo_LAPIG\phase3_changepoint"

# Effective analysis period (1985 and 2024 excluded — see notebook header)
YEAR_START = 1986
YEAR_END   = 2023

# Minimum valid observations required for changepoint detection
MIN_VALID_OBS = 10

# Algorithm settings
# Mode A: Binseg n_bkps=1 — Binseg accepts n_bkps; Pelt does not
# Mode B: Pelt pen=3.0 — validated empirically (see cell 9 and notebook header)
PELT_MODEL = "rbf"  # radial basis function cost — robust for EII in [0, 1]
PELT_PEN_B = 3.0    # validated fixed penalty for Mode B

---
## 2. Dependencies

In [ ]:
import importlib, subprocess, sys

for pkg in ["ruptures", "pandas", "numpy", "geopandas", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"OK {pkg}")

---
## 3. Load EII time series

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import ruptures as rpt
import matplotlib.pyplot as plt

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Loading EII annual dataset...")
df_eii = pd.read_csv(EII_CSV)
print(f"  Shape: {df_eii.shape}")

# Select columns for the effective analysis period only
years    = list(range(YEAR_START, YEAR_END + 1))
eii_cols = [c for c in df_eii.columns
            if any(str(y) in c for y in years) and "eii" in c.lower()]
eii_cols = sorted(eii_cols, key=lambda c: int(c[-4:]))

years_found   = sorted(set(int(c[-4:]) for c in eii_cols))
years_missing = [y for y in years if y not in years_found]
if years_missing:
    print(f"WARNING: missing years: {years_missing}")
else:
    print(f"  Years: {YEAR_START}-{YEAR_END} ({len(years)} years, all present)")

ids = df_eii["ID_UNICO"].values
X   = df_eii[eii_cols].values.astype(float)  # shape: (11500, 38)

print(f"  Matrix shape: {X.shape} ({X.shape[0]:,} cells x {X.shape[1]} years)")
print(f"  NaN rate: {np.isnan(X).mean()*100:.2f}%")

---
## 4. Change point detection

**Mode A:** Binseg with n_bkps=1 — identifies the single dominant structural
break per cell. Used for spatial mapping and temporal characterization.

**Mode B:** Pelt with pen=3.0 — detects multiple breaks automatically.
Penalty validated empirically (see cell 9 and notebook header).

Both use RBF (radial basis function) cost, robust for bounded [0,1] data.

In [ ]:
import time

n_cells = X.shape[0]
n_years = X.shape[1]

results = {
    "ID_UNICO":    ids,
    "valid":       np.zeros(n_cells, dtype=bool),
    "n_valid_obs": np.zeros(n_cells, dtype=int),
    "cp_year_A":   np.full(n_cells, np.nan),
    "eii_before":  np.full(n_cells, np.nan),
    "eii_after":   np.full(n_cells, np.nan),
    "delta_eii":   np.full(n_cells, np.nan),
    "cp_n_B":      np.zeros(n_cells, dtype=int),
    "cp_years_B":  [""] * n_cells,
}

print(f"Running change point detection on {n_cells:,} cells ({n_years} years each)...")
print(f"  Mode A: Binseg n_bkps=1 (dominant break)")
print(f"  Mode B: Pelt pen={PELT_PEN_B} (automatic, validated)")
t_start = time.time()

for i in range(n_cells):
    series = X[i, :]

    valid_mask = np.isfinite(series)
    n_valid    = valid_mask.sum()
    results["n_valid_obs"][i] = n_valid

    if n_valid < MIN_VALID_OBS:
        continue

    results["valid"][i] = True

    # Interpolate NaN for algorithm (NaN rare in rectangular domain)
    series_filled = pd.Series(series).interpolate(
        method="linear", limit_direction="both").values

    # ── Mode A: exactly 1 changepoint using Binseg ──────────────────
    # Note: Binseg.predict() accepts n_bkps; Pelt.predict() does not
    try:
        algo_A  = rpt.Binseg(model=PELT_MODEL).fit(series_filled)
        bkps_A  = algo_A.predict(n_bkps=1)   # [break_idx, n_years]
        brk_A   = bkps_A[0]                  # index of first element AFTER break

        results["cp_year_A"][i]  = years[brk_A - 1]
        results["eii_before"][i] = round(float(np.nanmean(series[:brk_A])), 4)
        results["eii_after"][i]  = round(float(np.nanmean(series[brk_A:])), 4)
        results["delta_eii"][i]  = round(
            results["eii_after"][i] - results["eii_before"][i], 4)
    except Exception:
        pass

    # ── Mode B: automatic with validated fixed penalty ───────────────
    try:
        algo_B     = rpt.Pelt(model=PELT_MODEL).fit(series_filled)
        bkps_B     = algo_B.predict(pen=PELT_PEN_B)
        breaks_B   = [b for b in bkps_B if b < n_years]
        cp_years_B = [years[b - 1] for b in breaks_B]
        results["cp_n_B"][i]     = len(cp_years_B)
        results["cp_years_B"][i] = ";".join(str(y) for y in cp_years_B)
    except Exception:
        pass

elapsed = time.time() - t_start
print(f"\nDone in {elapsed:.0f}s ({elapsed/n_cells*1000:.1f} ms per cell)")
print(f"Valid cells: {results['valid'].sum():,} / {n_cells:,}")
print("\nFirst 5 cells (sample check):")
for i in range(5):
    print(f"  ID={ids[i]}  cp_A={results['cp_year_A'][i]}  "
          f"delta={results['delta_eii'][i]}  "
          f"cp_n_B={results['cp_n_B'][i]}  "
          f"cp_years_B={results['cp_years_B'][i]}")

---
## 5. Export results

In [ ]:
df_cp = pd.DataFrame({
    "ID_UNICO":    results["ID_UNICO"],
    "valid":       results["valid"],
    "n_valid_obs": results["n_valid_obs"],
    "cp_year_A":   results["cp_year_A"],
    "eii_before":  results["eii_before"],
    "eii_after":   results["eii_after"],
    "delta_eii":   results["delta_eii"],
    "cp_n_B":      results["cp_n_B"],
    "cp_years_B":  results["cp_years_B"],
})

df_cp["cp_year_A"] = df_cp["cp_year_A"].where(
    df_cp["valid"], np.nan).astype("Int64")

out_path = os.path.join(OUTPUT_FOLDER, "eii_changepoints.csv")
df_cp.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"Saved: {out_path}")
print(f"Shape: {df_cp.shape}")
print(f"\nSummary:")
print(df_cp[["cp_year_A", "delta_eii", "cp_n_B"]].describe())

---
## 6. Mode A vs. Mode B comparison

If Mode B rarely detects more than 1 break, Mode A is well-justified.
If Mode B frequently detects 2 breaks, that is itself a result to report
(two distinct pulses of landscape pressure).

In [ ]:
valid = df_cp[df_cp["valid"]].copy()

print("=== Mode B — distribution of changepoint counts ===")
n_dist = valid["cp_n_B"].value_counts().sort_index()
for n, count in n_dist.items():
    pct = 100 * count / len(valid)
    print(f"  {n} changepoints: {count:,} cells ({pct:.1f}%)")

# Agreement where Mode B detected exactly 1
both_1 = valid[valid["cp_n_B"] == 1].copy()
if len(both_1) > 0:
    both_1["cp_year_B_first"] = both_1["cp_years_B"].apply(
        lambda x: int(x.split(";")[0]) if x else np.nan)
    agreement  = (both_1["cp_year_A"] == both_1["cp_year_B_first"]).mean()
    year_diff  = (both_1["cp_year_A"] - both_1["cp_year_B_first"]).abs()
    print(f"\nYear agreement (Mode A vs. Mode B, cells with exactly 1 break):")
    print(f"  Exact match:    {agreement*100:.1f}%")
    print(f"  Within +/-2yr:  {(year_diff <= 2).mean()*100:.1f}%")
    print(f"  Mean year diff: {year_diff.mean():.1f} years")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

n_dist.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_xlabel("Number of changepoints (Mode B)")
axes[0].set_ylabel("Number of cells")
axes[0].set_title("Distribution of changepoint counts\n(pen=3.0 automatic, Mode B)")
axes[0].tick_params(axis="x", rotation=0)

valid["cp_year_A"].dropna().astype(int).hist(
    bins=range(YEAR_START, YEAR_END + 2),
    ax=axes[1], color="steelblue", edgecolor="white")
axes[1].set_xlabel("Changepoint year (Mode A)")
axes[1].set_ylabel("Number of cells")
axes[1].set_title("Distribution of changepoint years\n(Binseg n_bkps=1, Mode A)")

plt.tight_layout()
fig_path = os.path.join(OUTPUT_FOLDER, "mode_comparison.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"\nFigure saved: {fig_path}")

---
## 7. Spatial maps

Three panels:
1. Year of dominant EII structural break (Mode A)
2. Magnitude of EII change at break (delta_eii)
3. Number of structural breaks per cell (Mode B)

In [ ]:
print("Loading grid for spatial mapping...")
gdf = gpd.read_file(GRID_SHAPEFILE)
if "ID_UNICO" not in gdf.columns:
    gdf["ID_UNICO"] = gdf.index.astype(int)

gdf_cp = gdf.merge(
    df_cp[["ID_UNICO", "cp_year_A", "delta_eii", "cp_n_B", "valid"]],
    on="ID_UNICO", how="left")

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

gdf_cp[gdf_cp["valid"]].plot(
    column="cp_year_A", ax=axes[0], cmap="RdYlBu_r",
    legend=True, legend_kwds={"label": "Changepoint year"},
    linewidth=0.1, edgecolor="none")
axes[0].set_title("Year of EII structural break\n(Mode A — 1 changepoint)", fontsize=10)
axes[0].axis("off")

vmax = gdf_cp["delta_eii"].abs().quantile(0.95)
gdf_cp[gdf_cp["valid"]].plot(
    column="delta_eii", ax=axes[1], cmap="RdBu",
    vmin=-vmax, vmax=vmax,
    legend=True, legend_kwds={"label": "delta EII (after - before)"},
    linewidth=0.1, edgecolor="none")
axes[1].set_title("Magnitude of EII change at breakpoint\n(delta EII = EII_after - EII_before)",
                   fontsize=10)
axes[1].axis("off")

gdf_cp[gdf_cp["valid"]].plot(
    column="cp_n_B", ax=axes[2], cmap="YlOrRd",
    legend=True, legend_kwds={"label": "N changepoints (BIC)"},
    linewidth=0.1, edgecolor="none")
axes[2].set_title("Number of EII structural breaks\n(Mode B — pen=3.0 automatic)",
                   fontsize=10)
axes[2].axis("off")

plt.suptitle("EII Change Point Detection — HEX-20, 1986-2023", fontsize=12)
plt.tight_layout()
map_path = os.path.join(OUTPUT_FOLDER, "eii_changepoints_map.png")
plt.savefig(map_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Map saved: {map_path}")

---
## 8. Summary statistics

In [ ]:
valid = df_cp[df_cp["valid"]].copy()

print("=== Change Point Detection Summary ===")
print(f"\nValid cells: {len(valid):,} / {len(df_cp):,}")
print(f"Analysis period: {YEAR_START}-{YEAR_END} ({YEAR_END-YEAR_START+1} years)")

print("\n--- Mode A (1 changepoint forced, Binseg) ---")
cp_years = valid["cp_year_A"].dropna().astype(int)
print(f"  Year range: {cp_years.min()}-{cp_years.max()}")
print(f"  Median: {int(cp_years.median())}  |  Mean: {cp_years.mean():.1f}")
print(f"  Delta EII mean: {valid['delta_eii'].mean():.4f}")
print(f"  Delta EII < 0 (decline): {(valid['delta_eii'] < 0).mean()*100:.1f}%")

print("\n  Changepoints by 5-year period:")
for y0 in range(YEAR_START, YEAR_END + 1, 5):
    y1  = min(y0 + 4, YEAR_END)
    sub = valid[(valid["cp_year_A"] >= y0) & (valid["cp_year_A"] <= y1)]
    if len(sub) == 0:
        continue
    print(f"    {y0}-{y1}: n={len(sub):4d} ({100*len(sub)/len(valid):.1f}%)  "
          f"delta_mean={sub['delta_eii'].mean():.3f}  "
          f"pct_neg={100*(sub['delta_eii']<0).mean():.0f}%")

print("\n--- Mode B (automatic, pen=3.0) ---")
for n in [0, 1, 2, 3]:
    label = "3+" if n == 3 else str(n)
    cnt   = (valid["cp_n_B"] >= 3).sum() if n == 3 else (valid["cp_n_B"] == n).sum()
    print(f"  {label} breaks: {cnt:,} ({100*cnt/len(valid):.1f}%)")
print(f"  Mean breaks per cell: {valid['cp_n_B'].mean():.2f}")

summary_path = os.path.join(OUTPUT_FOLDER, "changepoint_summary.csv")
pd.DataFrame([{
    "n_valid_cells": len(valid),
    "year_start": YEAR_START, "year_end": YEAR_END,
    "cp_year_median_A": int(cp_years.median()),
    "cp_year_mean_A": round(cp_years.mean(), 1),
    "delta_eii_mean": round(valid["delta_eii"].mean(), 4),
    "pct_decline": round((valid["delta_eii"] < 0).mean()*100, 2),
    "pct_0_breaks_B": round((valid["cp_n_B"]==0).mean()*100, 2),
    "pct_1_break_B":  round((valid["cp_n_B"]==1).mean()*100, 2),
    "pct_2_breaks_B": round((valid["cp_n_B"]==2).mean()*100, 2),
    "pct_3plus_B":    round((valid["cp_n_B"]>=3).mean()*100, 2),
}]).to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"\nSummary saved: {summary_path}")

---
## 9. Sensitivity check — penalty parameter (Mode B)

Tests pen=1.5, 3.0, and 6.0 on a 500-cell subsample to document
that pen=3.0 is at the appropriate inflection point.
Results support the penalty choice reported in the paper Methods section.

In [ ]:
np.random.seed(42)
valid_idx  = np.where(results["valid"])[0]
sample_idx = np.random.choice(valid_idx, min(500, len(valid_idx)), replace=False)

print("Sensitivity to penalty parameter (500-cell subsample)")
print("Tested around the validated primary value pen=3.0\n")

for pen_label, pen_val in [
    ("pen=1.5  (loose  — half of primary)",  1.5),
    ("pen=3.0  (primary — Mode B baseline)", 3.0),
    ("pen=6.0  (strict — double primary)",   6.0),
]:
    counts = []
    for i in sample_idx:
        s = X[i, :]
        sf = pd.Series(s).interpolate(method="linear",
                                      limit_direction="both").values
        try:
            algo  = rpt.Pelt(model=PELT_MODEL).fit(sf)
            bkps  = algo.predict(pen=pen_val)
            counts.append(len([b for b in bkps if b < n_years]))
        except Exception:
            counts.append(0)

    arr = np.array(counts)
    print(f"{pen_label}:")
    for n in [0, 1, 2, 3]:
        label_n = "3+" if n == 3 else str(n)
        cnt = int((arr >= 3).sum()) if n == 3 else int((arr == n).sum())
        print(f"  {label_n} breaks: {cnt:3d} cells ({100*cnt/len(arr):.1f}%)")
    print(f"  Mean: {arr.mean():.2f}\n")

print("Interpretation:")
print("  pen=1.5: over-segments  -> validates pen=3.0 is not too strict")
print("  pen=3.0: balanced       -> primary choice (documented in paper)")
print("  pen=6.0: under-segments -> validates pen=3.0 is not too loose")